# Arab Youth Space Hackathon: 813 Challenge - Data exploration with open access satellite datasets

## 🌍 Earth Observation Hackathon Notebook

Welcome! This notebook will help you explore satellite data for the Arab Youth Space Hackathon: 813 Challenge.

---

## What you will learn

- What satellite data is
- What STAC is and why it's important
- How to access satellite imagery without downloading files
- How to calculate simple indices like:
  - NDVI (vegetation health)
  - NDWI (water detection)

---

## What is satellite data?

Satellites continuously capture images of Earth. These images contain scientific information about:
- Vegetation (crops, forests)
- Water bodies
- Urban areas
- Temperature and climate

---

## What is STAC?

STAC = **SpatioTemporal Asset Catalog**

Think of STAC as:
👉 A **search engine for satellite data**

Instead of downloading data manually, STAC allows you to:
- Search by location (bounding box)
- Filter by time
- Select datasets quickly
- Access data directly from the cloud

---

## What are COGs?

COG = **Cloud Optimized GeoTIFF**

This means:

✅ You can read only the part of the image you need  
✅ No need to download entire files  
✅ Faster and scalable  

---

Let's get started 🚀

## 📖 Glossary -- Key Terms Used in This Notebook

New to satellite data? Here's a quick reference for terms you'll see throughout this notebook.

| Term | Meaning |
|---|---|
| **STAC** | *SpatioTemporal Asset Catalog* -- a standard way to search satellite imagery by location, date, and other filters, like a search engine for satellite data. |
| **COG** | *Cloud Optimized GeoTIFF* -- an image file format that lets you read just the part you need directly from the cloud, without downloading the whole file. |
| **Band** | A single layer of an image captured at a specific range of wavelengths (e.g. "red", "near-infrared"). A normal photo has 3 bands (red, green, blue); satellites often capture many more. |
| **Multispectral vs. hyperspectral** | Multispectral sensors (like Sentinel-2) capture a handful of bands (~10). Hyperspectral sensors (like Planet Tanager) capture hundreds of narrow, contiguous bands -- much more detail about what's on the ground. |
| **Reflectance** | The fraction of sunlight a surface reflects back to the satellite, from 0 (absorbs everything) to 1 (reflects everything). It's the physical quantity most vegetation/water indices are built from. |
| **Bounding box (bbox)** | The rectangular area you want to search or display, given as `[min longitude, min latitude, max longitude, max latitude]`. |
| **CRS / EPSG code** | *Coordinate Reference System* -- defines how coordinates map to real locations on Earth. `EPSG:4326` is the standard latitude/longitude system used throughout this notebook. |
| **HDF5** | *Hierarchical Data Format v5* -- a file format for storing large, multi-dimensional scientific datasets (used here for Tanager's full 426-band hyperspectral cube). |
| **Index (e.g. NDVI, NDWI)** | A simple formula combining two or more bands to highlight a specific feature (vegetation, water, etc.), e.g. `NDVI = (NIR - Red) / (NIR + Red)`. |

### 📚 Further Reading
- [STAC specification](https://stacspec.org/)
- [Microsoft Planetary Computer docs](https://planetarycomputer.microsoft.com/docs/overview/about)
- [Sentinel-2 mission reference (ESA)](https://sentiwiki.copernicus.eu/web/s2-mission)
- [Planet Tanager overview](https://www.planet.com/products/hyperspectral/)
- [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) and [xarray docs](https://docs.xarray.dev/) if you're new to working with array data in Python

---

In [ ]:
%pip install pystac-client planetary-computer stackstac rasterio xarray matplotlib geopandas leafmap h5py ipywidgets rioxarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.2/669.2 kB 17.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 49.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 23.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 11.0 MB/s eta 0:00:00
   ━

In [ ]:
%pip install --upgrade certifi requests==2.32.4 urllib3 leafmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


### ⚠️ Troubleshooting

- **Just ran the `%pip install` cells above?** Restart the kernel (*Kernel -> Restart*) before running the rest of the notebook -- otherwise Python may keep using the old, already-imported versions of these packages.
- **Map not showing up?** Interactive maps (via `leafmap`) need a live Jupyter kernel -- they won't render in a plain script or some static viewers. Re-run the cell, or try `jupyter notebook`/`jupyter lab` locally, or Google Colab.
- **Getting a 403/permission error loading imagery?** Microsoft Planetary Computer requires data URLs to be "signed" before they can be accessed -- this notebook already handles that for you (see `modifier=planetary_computer.sign_inplace` below), but if you copy code out of here into your own script, keep that part.
- **Map errors when moving between the Sentinel-2 and Tanager sections?** These two sections use different map tile servers. This notebook resets the tile server endpoint where needed, but if you jump around out of order you may need to re-run the nearest `os.environ["TITILER_ENDPOINT"] = ...` cell for the section you're in.

---

In [ ]:
# Import libraries

import pystac_client
import planetary_computer
import stackstac
import matplotlib.pyplot as plt
import numpy as np

import rioxarray
import leafmap
from IPython.display import display
import h5py
import io

In [ ]:
# Titiler endpoint

import os

os.environ["TITILER_ENDPOINT"] = "planetary-computer"
#titiler_endpoint="pc"
titiler_endpoint="planetary-computer"


## Step 1 – Define Area of Interest (AOI)

We need to tell the system **where on Earth to look**.

We define a bounding box:
[min longitude, min latitude, max longitude, max latitude]

We also define a time range

This example uses a region in the UAE.

In [ ]:
bbox = [54.0, 24.0, 55.0, 25.0]  # [min lon, min lat, max lon, max lat]
time_range = "2025-01-01/2025-03-01"

# 1. Initialize map
m = leafmap.Map(center=[24.5, 54.5], zoom=8)

# 2. Add the bounding box as a layer
# layer_name allows you to toggle it on/off in the layer control
m.add_bbox(bbox, layer_name="AOI Bounding Box", color="red", fill_opacity=0.1)

# 3. Optional: Zoom the map directly to the extent of your bbox
m.zoom_to_bounds(bbox)

m

Map(center=[24.5, 54.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

## Step 2 – Connect to Sentinel-2 STAC

We connect to the **Planetary Computer STAC API**, a free, cloud-hosted catalog of satellite data operated by Microsoft that hosts Sentinel-2 (and many other open datasets).

### What is Sentinel-2?

**Sentinel-2** is an Earth observation mission run by the **European Space Agency (ESA)** as part of the EU's **Copernicus programme**. It's actually a pair of twin satellites — Sentinel-2A and Sentinel-2B — flying in the same orbit but offset by 180°, so together they image almost any point on Earth every **~5 days**.

Key things to know about it:
- **Multispectral**, not hyperspectral — it captures **13 spectral bands** across the visible, near-infrared, and shortwave-infrared. That's much coarser spectrally than the **426-band hyperspectral** Tanager data you'll use later in this notebook, but it's still very useful and far more frequent.
- **Spatial resolution**: 10 m for the visible/near-infrared bands (what you'll use for RGB images and NDVI), 20 m for red-edge/SWIR bands, 60 m for atmospheric-correction bands.
- **Free and fully open** — no cost, no approval process, available to anyone.
- **Global, frequent coverage** — roughly every 5 days anywhere on Earth thanks to the two-satellite constellation.
- **Common uses**: crop monitoring, deforestation and land-cover change, flood and wildfire mapping, water quality, and urban expansion.

We'll use the `sentinel-2-l2a` collection specifically — "L2A" means **Level-2A**, i.e. the imagery has already been atmospherically corrected to *surface reflectance*, so it's ready to analyze straight away.

In [ ]:
# modifier=planetary_computer.sign_inplace attaches short-lived access tokens
# to each item's asset URLs -- required to actually read Planetary Computer data.
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

## Step 3 - Search for satellite imagery

We search for images using:
- Bounding box (AOI)
- Date range
- Cloud cover filter

This avoids downloading unnecessary data.


In [ ]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    #datetime="2025-01-01/2025-03-01",
    datetime = time_range,
    query={"eo:cloud_cover": {"lt": 20}}, # we can set a cloud cover % treshold to filter out cloudy images
)

items = list(search.get_items())
len(items)

70

### Inspecting the retrieved list of STAC images

Using a GeoDataFrame is the standard professional approach for Earth Observation workflows. It allows you to:

- Visualize Coverage: You can plot the footprints of all available satellite scenes to see which ones overlap your area of interest.

- Filter Metadata: You can easily sort by eo:cloud_cover, datetime, or mgrs:utm_zone using standard Pandas filtering:

In [ ]:
import pystac
import geopandas

# 1. Convert the list of items into an ItemCollection
item_collection = pystac.ItemCollection(items)

# 2. Now convert the ItemCollection to a GeoDataFrame
# This method extracts the geometry and properties automatically
df = geopandas.GeoDataFrame.from_features(
    item_collection.to_dict(),
    crs="epsg:4326"
)

# Preview the results
print(df.head())

                                            geometry  \
0  POLYGON ((54.92275 25.30152, 54.91902 25.28754...   
1  POLYGON ((54.41689 23.64506, 54.2398 23.69331,...   
2  POLYGON ((54.91283 25.26431, 54.87966 25.13976...   
3  POLYGON ((52.98622 25.30335, 54.07586 25.28464...   
4  POLYGON ((54.41689 23.64506, 54.2398 23.69331,...   

                      datetime     platform instruments s2:mgrs_tile  \
0  2025-03-01T06:57:29.024000Z  Sentinel-2B       [msi]        40RBN   
1  2025-03-01T06:57:29.024000Z  Sentinel-2B       [msi]        40QBM   
2  2025-03-01T06:57:29.024000Z  Sentinel-2B       [msi]        39RZH   
3  2025-03-01T06:57:29.024000Z  Sentinel-2B       [msi]        39RYH   
4  2025-03-01T06:57:29.024000Z  Sentinel-2B       [msi]        39QZG   

  constellation                                      s2:granule_id  \
0    Sentinel 2  S2B_OPER_MSI_L2A_TL_2BPS_20250301T091446_A0417...   
1    Sentinel 2  S2B_OPER_MSI_L2A_TL_2BPS_20250301T091446_A0417...   
2    Sentinel 2  S2B


## Step 4 – Load Data from the Cloud

We now load imagery directly from cloud-hosted COGs.

We select bands:
- B04 → Red
- B03 → Green
- B02 → Blue

Together, these form an RGB image.
``


In [ ]:

data = stackstac.stack(
    items,
    assets=["B04", "B03", "B02"],
    resolution=10,
    bounds_latlon=bbox,
    epsg=4326
).median(dim="time")


## Step 5 – Visualize Sentinel-2 RGB Image

We display the satellite image as a normal photo.
``

In [ ]:
# 1. Initialize map centered over the UAE area
bbox = [54.0, 24.0, 55.0, 25.0]  # [min lon, min lat, max lon, max lat]
m = leafmap.Map(center=[24.5, 54.5], zoom=8)

# 2. Query the STAC API catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-01-01/2025-03-01",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.get_items())
first_item = items[0]

# 3. Add the layer using leafmap's native STAC interpreter with the "pc" shortcode
m.add_stac_layer(
    collection="sentinel-2-l2a",
    item=first_item.id,
    assets=["visual"],       # Serves pre-compiled RGB. Use ["B04", "B03", "B02"] for raw bands
    titiler_endpoint="pc",   # Tells leafmap to route requests through Planetary Computer's tiler smoothly
    name="Sentinel-2 RGB"
)

m

Map(center=[24.5, 54.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

thumbnail = item.assets["thumbnail"].href
print(thumbnail)

## ✅ Next Steps

Now you can:

- Change the AOI to your country or region
- Try different time ranges
- Compare multiple dates (change detection)

---

## 💡 Tips

- Start small (small AOI)
- Avoid downloading large datasets
- Experiment with indices

---

You are now ready to build your hackathon solution 🚀
``

# 🌊 Planet Tanager Coastal Water Bodies Explorer

This notebook helps you explore **Planet’s Tanager hyperspectral dataset**.

---

## What You Will Learn

- What hyperspectral satellite data is
- How to query a STAC catalog
- How to explore metadata
- How to visualize quicklook images
- How to derive simple insights (time, location, coverage)

---

## What is Tanager?

Tanager is a **hyperspectral satellite mission** providing:
- High spectral resolution (many bands)
- Detailed water and coastal analysis capability

Unlike Sentinel‑2:
- It captures many more spectral bands
- It enables detailed chemical/biophysical analysis

We connect to Planet’s open STAC collection.

---

## What is STAC?

STAC (SpatioTemporal Asset Catalog) lets you:
- Search satellite data
- Access metadata
- Retrieve imagery

👉 Think: “Google for satellite data”


## Step 1 – Connect to the STAC Collection

Below we load a handful of specific Tanager scenes by their **item ID** from Planet's `coastal-water-bodies` collection. These IDs were picked ahead of time as good examples for this walkthrough -- they aren't anything special, just five scenes that happen to exist in this collection.

**Want to find other scenes?** Every Planet STAC collection has a `collection.json` that lists all of its items. You can either:
- Browse it in your browser: https://www.planet.com/data/stac/browser/tanager-core-imagery/coastal-water-bodies/collection.json
- Or fetch it in Python and list the item IDs programmatically:
  ```python
  import requests
  col_url = "https://www.planet.com/data/stac/tanager-core-imagery/coastal-water-bodies/collection.json"
  col = requests.get(col_url).json()
  item_links = [l for l in col.get("links", []) if l.get("rel") == "item"]
  print(f"Found {len(item_links)} scenes")
  for link in item_links[:10]:
      print(link["href"].split("/")[-1].replace(".json", ""))
  ```
  Each printed ID can be dropped straight into `item_ids` below. The same pattern works for any other Tanager collection -- just swap `coastal-water-bodies` for e.g. `agriculture`, `urban`, `fire`, or `GHG-plumes`.

In [ ]:
import requests

# Example scene IDs from the coastal-water-bodies collection -- see the
# markdown cell above for how to browse or fetch the full list of IDs.
item_ids = [
    "20250223_165546_32_4001",
    "20250225_153627_16_4001",
    "20250302_030003_92_4001",
    "20250311_061550_53_4001",
    "20250406_170447_47_4001"
]

items = []

for iid in item_ids:
    url = f"https://www.planet.com/data/stac/tanager-core-imagery/coastal-water-bodies/{iid}/{iid}.json"
    items.append(requests.get(url).json())

print(f"Loaded {len(items)} scenes")

Loaded 5 scenes


## Step 2 – Inspect Metadata

In [ ]:
item = items[0]

print("ID:", item["id"])
print("Date:", item["properties"]["datetime"])
print("Bounding Box:", item["bbox"])

ID: 20250223_165546_32_4001
Date: 2025-02-23T16:55:46.321891Z
Bounding Box: [-88.56425877863482, 13.139414553891845, -88.34746243694369, 13.316052417927148]


## Step 3 – View Full Metadata

Each image includes detailed metadata.

In [ ]:
import json

print(json.dumps(item["properties"], indent=2))

{
  "cloud_percent": 0,
  "collection_mode": "standard_sensitivity",
  "constellation": "Tanager",
  "datetime": "2025-02-23T16:55:46.321891Z",
  "description": "All data products from Tanager-1 for 20250223_165546_32_4001 near Usulut\u00e1n, 3401, El Salvador",
  "gsd": 32.85,
  "instruments": [
    "4001"
  ],
  "license": "CC-BY-4.0",
  "light_haze_percent": 0,
  "location_description": "Usulut\u00e1n, 3401, El Salvador",
  "notes": null,
  "platform": "Planet",
  "plume_provider_ids": null,
  "quality_category": "standard",
  "title": "TanagerScene 20250223_165546_32_4001 Core Imagery",
  "view:azimuth": 263.4,
  "view:off_nadir": 11.2,
  "view:sun_azimuth": 141.3,
  "view:sun_elevation": 61.2
}


## Step 4 – Display Image

Quicklook images allow fast visualization.


In [ ]:
from IPython.display import Image

thumbnail = item["assets"]["thumbnail"]["href"]
Image(url=thumbnail)

## Step 5 – Display Full Resolution Image (COG)

Instead of using the thumbnail, we load the actual **Cloud Optimized GeoTIFF (COG)**.

COGs allow:
- Partial loading from cloud
- Efficient visualization
- Real satellite data access

We will display the `analytic` or `visual` asset.
``

In [ ]:
# The Tanager COGs below are public and served via a different tile endpoint than
# Planetary Computer -- reset it here so the map renders correctly in this section.
os.environ["TITILER_ENDPOINT"] = "https://titiler.xyz"

In [ ]:
# Create map (we'll zoom to the scene after loading)
m = leafmap.Map(center=[24.2, 55.7], zoom=7)

# Pick the real COG GeoTIFF visual asset
asset_key = "ortho_visual"
cog_url = item["assets"][asset_key]["href"]

print("Using asset:", asset_key)
print("COG URL:", cog_url)

# Zoom to item bbox (recommended so you actually see it)
bbox = item["bbox"]  # [minx, miny, maxx, maxy]
m.fit_bounds([[bbox[1], bbox[0]], [bbox[3], bbox[2]]])

# Preferred method for COGs (fast + good rendering)
if hasattr(m, "add_cog_layer"):
    #m.add_cog_layer(cog_url, name=f"Tanager {asset_key}"),
    m.add_cog_layer(cog_url, name=f"Tanager {asset_key}", titiler_endpoint="https://titiler.xyz"),
    titiler_endpoint="pc"  # leafmap's default public titiler
else:
    # fallback for older leafmap versions
    m.add_raster(cog_url, layer_name=f"Tanager {asset_key}")

display(m)

Using asset: ortho_visual
COG URL: https://storage.googleapis.com/open-cogs/planet-stac/tanager1-release2-core-imagery/ortho_visual/20250223_165546_32_4001_ortho_visual.tif


Map(center=[-88.45589371743637, 13.227770535956875], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Step 6 – Geographical Coverage

### Let's plot the image bounding box so we get a good understanding of the image extent

In [ ]:
import geopandas as gpd
from shapely.geometry import box

bbox = item["bbox"]
geom = box(*bbox)

gdf = gpd.GeoDataFrame({"geometry": [geom]}, crs="EPSG:4326")

m.add_gdf(gdf, layer_name="Scene Footprint", style={"color": "red", "fillOpacity": 0})
m

Map(center=[13.227770535956875, -88.45589371743637], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### Compute spatial analytics from Tanager hdf5 file

- For ocean/coastal monitoring, hyperspectral indices usually rely on specific wavelengths, and Tanager’s true spectral data is typically in:
basic_sr_hdf5 or ortho_sr_hdf5 (surface reflectance, 0–1)

In [ ]:
sr_key = "ortho_sr_hdf5" if "ortho_sr_hdf5" in item["assets"] else "basic_sr_hdf5"
sr_url = item["assets"][sr_key]["href"]
print("Using SR HDF5:", sr_key)
print(sr_url)

Using SR HDF5: ortho_sr_hdf5
https://storage.googleapis.com/open-cogs/planet-stac/tanager1-release2-core-imagery/ortho_sr_hdf5/20250223_165546_32_4001_ortho_sr_hdf5.h5


In [ ]:
%pip install -U h5py

### Analytics option A (optional, not recommended): Stream hyperspectral data directly into memory

⚠️ **Skip this and use the next section instead.** Tanager surface-reflectance HDF5 files are typically **1–5 GB**. The `requests.get(sr_url).content` call below loads the *entire* file into memory in one go before you can read anything from it -- for a multi-GB file over a typical connection, this can take a very long time and use a lot of RAM.

**Use "Download HDF5 image once and then compute statistics" below instead** -- it streams the download to disk in small chunks, shows progress, and skips re-downloading if the file already exists locally. That's the approach the rest of this notebook (and all the theme notebooks) uses.

The cell below is left here, commented out, purely as a reference for the in-memory approach -- it can be handy if you're working with a very small scene and want to avoid writing anything to disk.

In [ ]:
# import h5py, io
#
# # Download into memory (small scenes ok). If large, download to disk with streaming.
# data = requests.get(sr_url).content
# f = h5py.File(io.BytesIO(data), "r")
#
# # Inspect available groups/datasets
# def walk_h5(name, obj):
#     if isinstance(obj, h5py.Dataset):
#         print("DATASET:", name, obj.shape, obj.dtype)
#     elif isinstance(obj, h5py.Group):
#         pass
#
# f.visititems(walk_h5)

### Download HDF5 image once and then compute statistics

### 1) Download the HDF5 file locally
##### (May take some time and is dependent on your internet speed... so get a coffee)

In [ ]:
# Pick the surface reflectance HDF5 asset from the STAC item
sr_key = "ortho_sr_hdf5" if "ortho_sr_hdf5" in item["assets"] else "basic_sr_hdf5"
sr_url = item["assets"][sr_key]["href"]

print("Using SR asset:", sr_key)
print("Download URL:", sr_url)

# Save locally using the filename from the URL
local_path = os.path.basename(sr_url)

if not os.path.exists(local_path):
    print("Downloading file...")
    with requests.get(sr_url, stream=True) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                if chunk:
                    f.write(chunk)
    print("Saved to:", local_path)
else:
    print("File already exists:", local_path)

Using SR asset: ortho_sr_hdf5
Download URL: https://storage.googleapis.com/open-cogs/planet-stac/tanager1-release2-core-imagery/ortho_sr_hdf5/20250223_165546_32_4001_ortho_sr_hdf5.h5
Saved to: 20250223_165546_32_4001_ortho_sr_hdf5.h5


### 2) Read masks and compute quick scene-quality statistics


In [ ]:
root = "HDFEOS/GRIDS/HYP/Data Fields"

with h5py.File(local_path, "r") as f:
    cloud = f[f"{root}/beta_cloud_mask"][:]      # (655, 785)
    nodata = f[f"{root}/nodata_pixels"][:]       # (655, 785)
    cirrus = f[f"{root}/beta_cirrus_mask"][:]    # (655, 785)

valid = (nodata == 0) & (cloud == 0) & (cirrus == 0)

valid_pct = valid.mean() * 100
cloud_pct = (cloud != 0).mean() * 100
cirrus_pct = (cirrus != 0).mean() * 100
nodata_pct = (nodata != 0).mean() * 100

print(f"Valid pixels:  {valid_pct:.1f}%")
print(f"Cloud pixels:  {cloud_pct:.1f}%")
print(f"Cirrus pixels: {cirrus_pct:.1f}%")
print(f"NoData pixels: {nodata_pct:.1f}%")

NameError: name 'h5py' is not defined

### 3) Extract band wavelengths from the STAC metadata
This is a very useful trick: the STAC item asset metadata often already contains the spectral band center wavelengths, so you don’t need to guess.


In [ ]:
# Extract spectral wavelengths from the STAC asset metadata
#bands_meta = item["assets"][sr_key].get("eo:bands", [])
bands_meta = item["assets"][sr_key].get("bands", [])
#bands_meta = item["assets"][sr_key].get("raster:bands", [])

# Keep only true spectral bands (ignore mask/geometry metadata entries)
#spectral_bands = [b for b in bands_meta if "center_wavelength" in b]
spectral_bands = [b for b in bands_meta if "eo:center_wavelength" in b]

#wavelengths_um = np.array([b["center_wavelength"] for b in spectral_bands], dtype=float)
wavelengths_um = np.array([b["eo:center_wavelength"] for b in spectral_bands], dtype=float)
wavelengths_nm = wavelengths_um * 1000.0  # convert micrometers -> nanometers

print("Number of spectral bands from STAC:", len(wavelengths_nm))
print("First 10 wavelengths (nm):", np.round(wavelengths_nm[:10], 2))
print("Last 10 wavelengths (nm):", np.round(wavelengths_nm[-10:], 2))

### Check consistency
The hyperspectral cube you found is:

surface_reflectance (426, 655, 785)

So we should verify the wavelength vector length matches the band dimension:

In [ ]:
with h5py.File(local_path, "r") as f:
    sr_shape = f[f"{root}/surface_reflectance"].shape

print("Surface reflectance cube shape:", sr_shape)

if sr_shape[0] != len(wavelengths_nm):
    print("⚠️ Warning: band count and wavelength count differ.")
else:
    print("✅ Wavelength count matches cube band dimension.")

### 4) Helper functions for band selection
We’ll choose the band nearest to a target wavelength in nanometers.

In [ ]:
def nearest_band_idx(wavelengths_nm, target_nm):
    return int(np.argmin(np.abs(wavelengths_nm - target_nm)))

def print_band_choice(target_nm, wavelengths_nm):
    idx = nearest_band_idx(wavelengths_nm, target_nm)
    actual = wavelengths_nm[idx]
    print(f"Target {target_nm:.0f} nm -> band {idx} ({actual:.2f} nm)")
    return idx

### 5) Pick useful wavelengths for marine/coastal monitoring
These are practical choices:

- NDVI → coastal vegetation / mangroves

    - Red ≈ 665 nm
    - NIR ≈ 800 nm


- NDWI → water detection

    - Green ≈ 560 nm
    - NIR ≈ 860 nm


- NDCI → chlorophyll / algae bloom proxy

    - Red ≈ 665 nm
    - Red-edge ≈ 708 nm


- Turbidity proxy (NDTI style)

    - Red ≈ 665 nm
    - Green ≈ 560 nm

In [ ]:
b560 = print_band_choice(560, wavelengths_nm)
b665 = print_band_choice(665, wavelengths_nm)
b708 = print_band_choice(708, wavelengths_nm)
b800 = print_band_choice(800, wavelengths_nm)
b860 = print_band_choice(860, wavelengths_nm)

### 6) Load only the few required bands (fast)
This is the key speed trick:
👉 we do not read the full 426-band cube, only the bands needed for the indices.

In [ ]:
with h5py.File(local_path, "r") as f:
    sr = f[f"{root}/surface_reflectance"]

    R560 = sr[b560, :, :].astype("float32")
    R665 = sr[b665, :, :].astype("float32")
    R708 = sr[b708, :, :].astype("float32")
    R800 = sr[b800, :, :].astype("float32")
    R860 = sr[b860, :, :].astype("float32")

Apply the valid mask:

In [ ]:
R560[~valid] = np.nan
R665[~valid] = np.nan
R708[~valid] = np.nan
R800[~valid] = np.nan
R860[~valid] = np.nan

### 7) Compute indices
#### *NDVI - Normalized Difference Vegetation Index*

Useful for:

- mangroves
- coastal vegetation
- wetland vegetation

Good for:

- mangrove condition
- saltmarsh / coastal vegetation health
- distinguishing vegetation from bare/wet surfaces

In [ ]:
NDVI = (R800 - R665) / (R800 + R665 + 1e-6)

#### NDWI - Normalized Difference Water Index
Useful for:
- water detection
- distinguishing water from vegetation/soil

In [ ]:
NDWI = (R560 - R860) / (R560 + R860 + 1e-6)

#### NDCI - Normalized Difference Chlorophyll Index
Useful for:

- chlorophyll proxy
- potential algal blooms / phytoplankton activity

In [ ]:
NDCI = (R708 - R665) / (R708 + R665 + 1e-6)

#### Turbidity proxy (NDTI style)
- A simple proxy for suspended material / turbidity:

In [ ]:
Turbidity = (R665 - R560) / (R665 + R560 + 1e-6)

### 8) Plot the indices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

im1 = axes[0,0].imshow(NDVI, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0,0].set_title("NDVI (Coastal Vegetation / Mangroves)")
axes[0,0].axis("off")
plt.colorbar(im1, ax=axes[0,0], fraction=0.046, pad=0.04)

im2 = axes[0,1].imshow(NDWI, cmap="Blues", vmin=-0.5, vmax=0.8)
axes[0,1].set_title("NDWI (Water Detection)")
axes[0,1].axis("off")
plt.colorbar(im2, ax=axes[0,1], fraction=0.046, pad=0.04)

im3 = axes[1,0].imshow(NDCI, cmap="YlGn", vmin=-0.2, vmax=0.4)
axes[1,0].set_title("NDCI (Chlorophyll / Bloom Proxy)")
axes[1,0].axis("off")
plt.colorbar(im3, ax=axes[1,0], fraction=0.046, pad=0.04)

im4 = axes[1,1].imshow(Turbidity, cmap="inferno", vmin=-0.3, vmax=0.3)
axes[1,1].set_title("Turbidity Proxy")
axes[1,1].axis("off")
plt.colorbar(im4, ax=axes[1,1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### 9) Print quick summary statistics


In [ ]:
def summarize_index(name, arr):
    print(f"{name}:")
    print(f"  min   = {np.nanmin(arr):.4f}")
    print(f"  max   = {np.nanmax(arr):.4f}")
    print(f"  mean  = {np.nanmean(arr):.4f}")
    print(f"  std   = {np.nanstd(arr):.4f}")
    print()

summarize_index("NDVI", NDVI)
summarize_index("NDWI", NDWI)
summarize_index("NDCI", NDCI)
summarize_index("Turbidity", Turbidity)

### 10 Mean spectrum for valid pixels
This gives a very useful scene-level “spectral fingerprint”

In [ ]:
stride = 8  # increase for faster speed, decrease for more detail

with h5py.File(local_path, "r") as f:
    sr = f[f"{root}/surface_reflectance"]

    # spatial sampling
    cube_sample = sr[:, ::stride, ::stride].astype("float32")  # (bands, y, x)
    mask_sample = valid[::stride, ::stride]

    cube_sample[:, ~mask_sample] = np.nan
    mean_spectrum = np.nanmean(cube_sample, axis=(1, 2))

plt.figure(figsize=(10, 4))
plt.plot(wavelengths_nm, mean_spectrum, lw=1.5)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Mean Surface Reflectance")
plt.title("Mean Spectrum (Sampled Valid Pixels)")
plt.grid(True, alpha=0.3)
plt.show()

## Water vapor bands
The reason we see negative values and flat lines around 1800 nm is because of water vapor bands.

Why Remove Water Vapor Bands?

Water vapor bands are regions of the spectrum (typically 1350–1450 nm and 1800–1950 nm) where the atmosphere absorbs almost all incoming light.

Why they must be removed:

Atmospheric Noise: Sensors record atmospheric opacity rather than ground reflectance, creating artificial "dips" in your data.

Skewed Statistics: Including these bands introduces heavy noise, which corrupts your mean spectral calculations.

Improved Accuracy: Removing these bands and negative values (artifacts from atmospheric correction) ensures your plot represents the true physical properties of the wetland.

## The other artifact: noisy spikes near 2500 nm
Look closely at the far-right edge of the plot above — beyond about 2450 nm the line gets erratic and spiky. This isn't a water vapor trough, it's a different problem: Tanager's SWIR2 detector array runs out to ~2499 nm, and its last channels sit right at the edge of the SWIR atmospheric window, where both solar irradiance and atmospheric transmission collapse. Since surface reflectance = radiance ÷ irradiance, dividing a tiny noisy signal by an even tinier reference blows up into spikes rather than showing real surface signal. This "detector edge" effect shows up in every imaging spectrometer (AVIRIS, PRISMA, EnMAP included), so we drop wavelengths beyond 2450 nm the same way we drop the water vapor troughs.

In [ ]:
## Let's remove the water vapor bands and the noisy SWIR-edge tail from the plot.

stride = 8 # increase for faster speed, decrease for more detail.

# Define atmospheric absorption windows to remove (e.g., 1350-1450nm and 1800-1950nm)
# Adjust these based on the specific wavelengths in your HDF5 file
def is_water_vapor_band(wl):
    return (1350 <= wl <= 1450) or (1800 <= wl <= 1950)

# Beyond ~2450 nm we're at the edge of the SWIR2 detector array — solar
# irradiance and atmospheric transmission both collapse there, and the
# detector's edge channels have the sensor's worst signal-to-noise ratio,
# so we drop that tail too (see "The other artifact" note above).
SWIR_EDGE_NM = 2450

def is_bad_band(wl):
    return is_water_vapor_band(wl) or wl > SWIR_EDGE_NM

with h5py.File(local_path, "r") as f:
    sr = f[f"{root}/surface_reflectance"]

    # 1. Filter out Water Vapor bands and the noisy SWIR-edge tail by index
    valid_band_indices = [i for i, wl in enumerate(wavelengths_nm) if not is_bad_band(wl)]

    # 2. Spatial sampling and apply band filter
    cube_sample = sr[valid_band_indices, ::stride, ::stride].astype("float32")
    mask_sample = valid[::stride, ::stride]

    # 3. Remove spatial nodata
    cube_sample[:, ~mask_sample] = np.nan

    # 4. REMOVE NEGATIVE VALUES: Set anything < 0 to NaN
    cube_sample[cube_sample < 0] = np.nan

    # Calculate mean ignoring NaNs
    mean_spectrum = np.nanmean(cube_sample, axis=(1, 2))

# Plotting
plt.figure(figsize=(10, 4))
clean_wavelengths = [wavelengths_nm[i] for i in valid_band_indices]

plt.plot(clean_wavelengths, mean_spectrum, lw=1.5, color='darkblue')
plt.xlabel("Wavelength (nm)")
plt.ylabel("Mean Surface Reflectance")
plt.title("Cleaned Mean Spectrum (No Water Vapor, No SWIR-Edge Noise, No Negatives)")
plt.grid(True, alpha=0.3)
plt.show()

---

## 🚀 Where to Go From Here

You've now walked through two very different Earth observation datasets in one notebook — Sentinel-2's frequent, moderate-resolution multispectral imagery, and Planet Tanager's detailed 426-band hyperspectral imagery. Here's how to turn that into a hackathon project.

### Pick a challenge theme

This quickstart is a general-purpose starting point. Head to the theme notebooks for a focused, ready-to-extend workflow:

- 🌾 **Precision Agriculture & Crop Intelligence** — `01_agriculture_crop_intelligence.ipynb`
- 🏙️ **Land Use & Land Cover Change** — `02_land_use_land_cover_change.ipynb`
- 💨 **Air Quality & GHG Plumes** — `03_air_quality_ghg_plumes.ipynb`
- 🔥 **Climate Disasters — Fire & Flood** — `04_climate_disasters_fire_flood.ipynb`
- 🌿 **Ecosystem Health & Blue Carbon** — `05_ecosystem_health_blue_carbon.ipynb`
- 💧 **Water Quality** — `06_ecosystem_health_water_qualityy.ipynb`

### Ideas to try with what you learned here

- **Combine Sentinel-2 and Tanager**: use Sentinel-2's frequent revisits to spot *when* something changed, then pull a Tanager scene for that date/location to understand *what* changed at high spectral detail.
- **Multi-date change detection**: query Sentinel-2 over the same AOI across several months and compare NDVI/NDWI over time.
- **Custom spectral indices**: Tanager's 426 bands let you build indices beyond NDVI/NDWI — try picking bands near known absorption features relevant to your theme (e.g. chlorophyll, moisture, minerals).
- **Swap in your own AOI**: replace the UAE bounding box with your own region of interest and re-run the notebook end to end.
- **Cloud and quality filtering**: experiment with the `eo:cloud_cover` threshold and the Tanager quality masks to see how they affect your results.
- **Build a simple classifier or index-based decision rule** on top of the bands you've already extracted here.

### Tips

- Start with a small AOI and a short time range — iterate fast before scaling up.
- Watch the HDF5 download size for Tanager scenes (typically 1–5 GB) — the notebooks skip re-downloading files that already exist locally.
- If your spectral plots look noisy, revisit the "Water Vapor Bands" section above — most artifacts come from atmospheric absorption regions, not the ground surface.

### 📊 Before You Submit: Think Like a Product, Not Just a Model

A working technical demo is only half of a hackathon submission. Before you submit your Proof of Concept, make sure you can also answer:

- **What real business problem does this solve, and for whom?** Who has this problem today, and what do they currently do about it without your solution?
- **How good is your model, really?** Include concrete validation statistics for your PoC — accuracy, precision/recall, RMSE, IoU, or a comparison against ground truth/known events — not just "it looks right on the map."
- **How will end users actually see your results?** You don't need to build a full frontend for the PoC, but you should be able to describe how your application would deliver value in practice — e.g. a dashboard, a REST API, an alerting/notification system, a scheduled report — and roughly what that would look like.

---

> 📖 Created by: [LinkedIn: Dr. Vincent Markiet – Space42](https://www.linkedin.com/in/vincentmarkiet/)

> 📖 Data: Copernicus Sentinel-2 data (free & open, ESA / EU Copernicus programme) · Planet Tanager data (CC-BY-4.0 © Planet Labs PBC)